# **Correction · TP5 | Modes de chiffrement**

On va utiliser des images BMP parce que les données binaires de ce format correspond directement aux données des pixels (ce n'est pas le cas pour d'autres formats).

La [structure de ces fichiers](https://upload.wikimedia.org/wikipedia/commons/7/75/BMPfileFormat.svg) consiste en une en-tête suivie des données de pixels. Le code `aesbmp.py` contient une classe `CipherFriendlyBmpImage` qui aide à manipuler ces fichiers.
La fonction `main()` de ce fichier contient l'exemple d'utilisation de cette classe qui a servi à chiffrer l'image `ciph-ecb.bmp` fournie.

L'objectif de ce TP est de déchiffrer cette image, pour ensuite comparer d'autres modes de chiffrement plus appropriés.

*Corrigé rédigé en l'absence de corrigé officiel. Les réponses ont été vérifiées en Python ; les cellules qui lisent un fichier compagnon (`aesbmp.py`, `ciph-ecb.bmp`, `cipher-cbc.bmp`, `iv-cbc.bin`) n'ont pas pu être exécutées ici et sont à lancer dans le dossier du TP.*

*Les fonctions `lire_bmp` / `ecrire_bmp` ci-dessous séparent l'en-tête des pixels sans dépendre de `CipherFriendlyBmpImage`, dont le code n'était pas fourni. La clé de `ciph-ecb.bmp` est supposée être `ALL THEM WITCHES` (celle des autres TP) : à vérifier dans `aesbmp.main()`.*

In [ ]:
import os
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding

from aesbmp import CipherFriendlyBmpImage

## 5.1. Mode de chiffrement ECB

Le mode de chiffrement utilisé est le [mode ECB](https://en.wikipedia.org/wiki/Block_cipher_mode_of_operation#Electronic_codebook_(ECB)) (Electronic Code Book), où chaque bloc est chiffré de la même manière.

![](chiffrement-ecb.svg)

En pratique, ce mode de chiffrement est vivement déconseillé, vu qu'il présente de nombreuses failles (par exemple les défis [12](https://cryptopals.com/sets/2/challenges/12), [13](https://cryptopals.com/sets/2/challenges/13) et [14](https://cryptopals.com/sets/2/challenges/14) de Cryptopals).

### 5.1.a. Observation de l'image chiffrée

**TODO**\
Observer l'image chiffrée `ciph-ecb.bmp` fournie. Arrivez-vous à distinguer des contours de l'image originale ? Est-ce un problème de sécurité ? Pourquoi ?

In [ ]:
from PIL import Image

Image.open("ciph-ecb.bmp")      # affiche l'image chiffrée dans le notebook

> **Oui, on distingue les contours** de l'image originale.
> En ECB, chaque bloc de 16 octets est chiffré seul, avec la même clé : deux blocs clairs identiques
> (un aplat de couleur) donnent deux blocs chiffrés identiques. Les zones unies restent donc unies,
> seule leur couleur change.
> C'est un **problème de sécurité** : la confidentialité n'est pas assurée, puisque la structure
> du message fuit sans qu'on ait besoin de la clé.

### 5.1.b. Déchiffrement de l'image

**TODO**  
Compléter la fonction `decrypt_aes_ecb` ci-dessous qui inverse la fonction `encrypt_aes_ecb` du fichier compagnon `aesbmp.py`, puis s'en servir pour déchiffrer l'image `ciph-ecb.bmp`.

In [ ]:
def decrypt_aes_ecb(data: bytes, key: bytes) -> bytes:
    """
    Déchiffre les données avec l'algorithme AES en mode ECB.

    Args:
        data (bytes): données à déchiffrer
        key (bytes): clé de chiffrement (16, 24 ou 32 bytes)
    Returns:
        bytes: données déchiffrées
    """
    decryptor = Cipher(algorithms.AES(key), modes.ECB()).decryptor()
    padded = decryptor.update(data) + decryptor.finalize()
    unpadder = padding.PKCS7(128).unpadder()        # le chiffrement a ajouté un padding PKCS#7
    return unpadder.update(padded) + unpadder.finalize()


def lire_bmp(chemin: str) -> tuple[bytes, bytes]:
    """Sépare l'en-tête (laissé en clair) des données de pixels (chiffrées)."""
    with open(chemin, "rb") as f:
        brut = f.read()
    debut_pixels = int.from_bytes(brut[10:14], "little")   # octets 10 à 13 de l'en-tête BMP
    return brut[:debut_pixels], brut[debut_pixels:]


def ecrire_bmp(chemin: str, entete: bytes, pixels: bytes) -> None:
    with open(chemin, "wb") as f:
        f.write(entete + pixels)


key = b"ALL THEM WITCHES"
entete, pixels_chiffres = lire_bmp("ciph-ecb.bmp")
ecrire_bmp("clear.bmp", entete, decrypt_aes_ecb(pixels_chiffres, key))
Image.open("clear.bmp")

## 5.2. Mode de chiffrement CBC

Un autre mode de chiffrement plus robuste (mais pas encore idéal) est le mode CBC :

![](chiffrement-cbc.svg)

Comme on le voit, celui-ci commence à partir d'une valeur initiale $IV$ aléatoire qui doit être de la même taille que les blocs. Ce mode est implémenté dans `cryptography`.

### 5.2.a. Chiffrer

**TODO**\
Compléter la fonction `encrypt_aes_cbc` ci-dessous.

In [ ]:
def encrypt_aes_cbc(data: bytes, key: bytes, iv: bytes) -> bytes:
    """
    Chiffre les données avec l'algorithme AES en mode CBC.

    Args:
        data (bytes): données à chiffrer
        key (bytes): clé de chiffrement (16, 24 ou 32 bytes)
        iv (bytes): vecteur d'initialisation (16 bytes)
    Returns:
        bytes: données chiffrées
    """
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv))
    padder = padding.PKCS7(128).padder()
    padded = padder.update(data) + padder.finalize()
    encryptor = cipher.encryptor()
    return encryptor.update(padded) + encryptor.finalize()

### 5.2.b. Test de la fonction

**TODO**\
Chiffrer des données vides (qui ne contiendront que le *padding*). Est-ce que le résultat contient l'IV ?

In [ ]:
data = b""
key = b"ALL THEM WITCHES"
iv = b"YELLOW SUBMARINE"

c = encrypt_aes_cbc(data, key, iv)
print(len(c), c.hex())          # 16 octets : un seul bloc, le padding 10 10 … 10 chiffré
print(iv in c)                  # False : l'IV n'apparaît pas dans le résultat

> Le résultat fait 16 octets : c'est le bloc de padding (16 octets de valeur `0x10`) chiffré.
> **Il ne contient pas l'IV.** Il faut donc transmettre l'IV à part (`iv.bin`). Il n'est pas secret,
> mais sans lui, le premier bloc est indéchiffrable.

### 5.2.c. Chiffrement d'une image

**TODO**
- Chiffrer l'image obtenue précédemment avec ce nouveau mode (on pourra utiliser `os.urandom` pour générer une IV).
- Sauvegarder le résultat dans un fichier `mycbc.bmp`.
- Sauvegarder l'IV dans un fichier `iv.bin` pour pouvoir déchiffrer l'image plus tard.

Est-ce que le résultat obtenu semble plus convaincant ?

In [ ]:
entete, pixels = lire_bmp("clear.bmp")
iv = os.urandom(16)                              # IV aléatoire, de la taille d'un bloc
ecrire_bmp("mycbc.bmp", entete, encrypt_aes_cbc(pixels, key, iv))
with open("iv.bin", "wb") as f:                  # l'IV n'est pas secret, mais il faut le garder
    f.write(iv)
Image.open("mycbc.bmp")

> **Oui, c'est bien plus convaincant** : l'image ressemble à du bruit uniforme.
> En CBC, chaque bloc est mélangé (XOR) au bloc chiffré précédent avant d'être chiffré :
> deux blocs clairs identiques donnent des blocs chiffrés différents.

### 5.2.d. Déchiffrement

**TODO**
- Compléter la fonction `decrypt_aes_cbc` ci-dessous qui inverse la fonction `encrypt_aes_cbc` précédente.
- S'en servir pour déchiffrer l'image `cipher-cbc.bmp` avec l'IV dans `iv-cbc.bin`.

In [ ]:
def decrypt_aes_cbc(data: bytes, key: bytes, iv: bytes) -> bytes:
    """
    Déchiffre les données avec l'algorithme AES en mode CBC.

    Args:
        data (bytes): données à déchiffrer
        key (bytes): clé de chiffrement (16, 24 ou 32 bytes)
    Returns:
        bytes: données déchiffrées
    """
    decryptor = Cipher(algorithms.AES(key), modes.CBC(iv)).decryptor()
    padded = decryptor.update(data) + decryptor.finalize()
    unpadder = padding.PKCS7(128).unpadder()
    return unpadder.update(padded) + unpadder.finalize()


with open("iv-cbc.bin", "rb") as f:
    iv_cbc = f.read()
entete, pixels_chiffres = lire_bmp("cipher-cbc.bmp")
ecrire_bmp("clear-cbc.bmp", entete, decrypt_aes_cbc(pixels_chiffres, key, iv_cbc))
Image.open("clear-cbc.bmp")

## Pour aller plus loin

### Comparaison entre le mode CBC "à la main" et celui qui utilise `cryptography`

In [ ]:
def xor(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))


def encrypt_aes_cbc_handmade(data: bytes, key: bytes, iv: bytes):
    """
    Chiffre les données avec l'algorithme AES en mode CBC, en utilisant
    `cryptography` seulement avec le mode ECB.

    Args:
        data (bytes): données à chiffrer
        key (bytes): clé de chiffrement (16, 24 ou 32 bytes)
        iv (bytes): vecteur d'initialisation (16 bytes)
    Returns:
        bytes: données chiffrées
    """
    cipher = Cipher(algorithms.AES(key), modes.ECB())
    padder = padding.PKCS7(128).padder()
    data = padder.update(data) + padder.finalize()
    encryptor = cipher.encryptor()
    precedent = iv                                   # c_0 = IV
    sortie = b""
    for i in range(0, len(data), 16):
        # c_i = AES(p_i XOR c_{i-1}) : chaque bloc dépend de tous les précédents
        precedent = encryptor.update(xor(data[i:i + 16], precedent))
        sortie += precedent
    return sortie + encryptor.finalize()


data = "Quelques données (texte) à chiffrer.".encode()
key = b"ALL THEM WITCHES"
iv = os.urandom(16)

ref_cbc = encrypt_aes_cbc(data, key, iv)
hand_cbc = encrypt_aes_cbc_handmade(data, key, iv)

print(ref_cbc == hand_cbc)